In [1]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0428 20:45:59.602000 14284 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# 1. Configuration
max_seq_length = 2048 # Gemma 4 supports up to 128K, but 4K is plenty for T&C paragraphs
dtype = None # None for auto-detection (Float16/Bfloat16)
load_in_4bit = True # Use 4-bit quantization to reduce memory usage


In [3]:
# %%
# 2. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # The pure-text powerhouse
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, 
)


==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce GTX 1650. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [4]:
# %%
# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # These remain standard for Llama
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
from datasets import load_dataset

# 4. Data Preparation (Mapping to Gemma 4 Chat Template)
# Since E2B/E4B are edge models, we use the standard turn-based template.
dataset = load_dataset("json", data_files="tnc_finetune_data.json", split="train")


In [ ]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Llama 3.2 Structure: <|begin_of_text|><|start_header_id|>system<|end_header_id|>...
        parts = [
            {"role": "system", "content": instruction},
            {"role": "user", "content": input},
            {"role": "assistant", "content": output}
        ]
        # Ensure we use add_generation_prompt=False during training
        text = tokenizer.apply_chat_template(parts, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)


In [7]:
# 5. Training Setup
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set to True for faster training with small sequences
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60, # Increase this for better convergence (e.g., 300-500)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)


Unsloth: Tokenizing ["text"]:   0%|          | 0/50 [00:00<?, ? examples/s]

In [8]:
# 6. Run Training
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 9 | Total steps = 60
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,2.917505
2,3.002694
3,2.937034
4,2.835661
5,2.612483
6,2.468519
7,2.248768
8,2.113061
9,1.965646
10,1.822657


KeyboardInterrupt: 

In [9]:
# 7. Save the Model
# Save LoRA adapters
model.save_pretrained("gemma4_tnc_lora")
tokenizer.save_pretrained("gemma4_tnc_lora")


Unsloth: Restored added_tokens_decoder metadata in gemma4_tnc_lora\tokenizer_config.json.


('gemma4_tnc_lora\\tokenizer_config.json',
 'gemma4_tnc_lora\\chat_template.jinja',
 'gemma4_tnc_lora\\tokenizer.json')

In [ ]:
model.save_pretrained_gguf("legal_shield_q4", tokenizer, quantization_method = "q4_k_m")


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in legal_shield_q4\tokenizer_config.json.


Found HuggingFace hub cache directory: C:\Users\singh\.cache\huggingface\hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]